# Fixing PADIM Tiling in Your Current Anomalib Version

> Practical guide to fix tiling bug in your current codebase

This notebook shows exactly how to fix the tiling issue for PADIM training in your current anomalib < 2.0 version.

## 🐛 The Bug in Your Current Code

I found the tiling bug in your codebase! Let me show you:

### Location: `be_vision_ad_tools/training/flexible_trainer.py`

**Line 579** has the WRONG import:

```python
❌ WRONG (Current code):
from anomalib.callbacks import TilingConfigurationCallback
```

**The Correct Import:**

```python
✅ CORRECT:
from anomalib.callbacks import TilerConfigurationCallback
```

**The difference**: `Tiler` vs `Tiling` (you wrote `Tiling` but it should be `Tiler`)

### Why This Breaks:

1. The class name is `TilerConfigurationCallback` (not `TilingConfigurationCallback`)
2. When you try to import the wrong name, Python will raise `ImportError`
3. Even if the import somehow passes, the callback won't work correctly

### Also in Your Code:

At **line 64**, you have the CORRECT import:

```python
✅ CORRECT (line 64):
from anomalib.callbacks import TilerConfigurationCallback
```

But then at **line 579** inside the function, you're importing the WRONG one! This means:
- The top-level import is correct
- But inside the `train_anomaly_model()` function, you're re-importing with the wrong name
- This causes tiling to fail when you enable it

## Step 1: Check Your Current Setup

In [ ]:
import anomalib
print(f"Anomalib version: {anomalib.__version__}")

# Check if we're using version < 2
version_parts = anomalib.__version__.split('.')
major_version = int(version_parts[0])

if major_version < 2:
    print(f"\n⚠️  You're using anomalib v{anomalib.__version__} (version 1.x)")
    print(f"   Tiling may have issues, but we can fix it!")
else:
    print(f"\n✅ You're using anomalib v{anomalib.__version__} (version 2.x+)")
    print(f"   Tiling should work correctly in this version.")

## Step 2: The CORRECT Import

Let's verify the correct import works:

In [ ]:
# Test CORRECT import
try:
    from anomalib.callbacks import TilerConfigurationCallback
    print("✅ CORRECT import works: TilerConfigurationCallback")
    print(f"   Class: {TilerConfigurationCallback}")
    print(f"   Module: {TilerConfigurationCallback.__module__}")
except ImportError as e:
    print(f"❌ Import failed: {e}")

print("\n" + "="*60 + "\n")

# Test WRONG import (this should fail)
try:
    from anomalib.callbacks import TilingConfigurationCallback
    print("⚠️  WRONG import somehow succeeded: TilingConfigurationCallback")
    print(f"   This shouldn't happen! Class: {TilingConfigurationCallback}")
except ImportError as e:
    print(f"❌ WRONG import failed (expected): {e}")
    print(f"   Error: Cannot import 'TilingConfigurationCallback'")
    print(f"   Correct name: 'TilerConfigurationCallback'")

## Step 3: Complete CORRECTED PADIM Training with Tiling

Here's the fixed version of your training code:

In [ ]:
#| export
import sys
import warnings
from pathlib import Path
from typing import Tuple, Optional, Dict, Any, List

# Core libraries
import numpy as np
import torch
from PIL import Image

# Anomalib imports - CORRECTED
import anomalib
from anomalib import TaskType
from anomalib.data.image.folder import Folder
from anomalib.engine import Engine
from anomalib.models import Padim
from anomalib.deploy import ExportType, TorchInferencer

# ✅ CORRECT import - use Tiler, not Tiling
from anomalib.callbacks import TilerConfigurationCallback

from lightning.pytorch.callbacks import ModelCheckpoint

# Suppress warnings
warnings.filterwarnings('ignore')

print(f"✅ All imports successful")
print(f"   Anomalib version: {anomalib.__version__}")
print(f"   Using: TilerConfigurationCallback (CORRECT)")

## Step 4: Fixed Training Function for PADIM with Tiling

This is your `train_anomaly_model()` function with the tiling bug fixed:

In [ ]:
#| export
def train_padim_with_tiling_fixed(
    data_root: Path,
    normal_dir: str = "good",
    abnormal_dir: str = "bad",
    class_name: str = "defect",
    backbone: str = "resnet18",
    layers: List[str] = None,
    n_features: int = 100,
    image_size: Tuple[int, int] = (512, 512),
    # Tiling parameters
    enable_tiling: bool = True,
    tile_size: Tuple[int, int] = (256, 256),
    stride: Tuple[int, int] = (128, 128),
    # Training parameters
    max_epochs: int = 50,
    train_batch_size: int = 16,
    eval_batch_size: int = 16,
    num_workers: int = 0,
    save_path: Path = None
) -> Dict[str, Any]:
    """
    Train PADIM model with CORRECTED tiling implementation.
    
    This function has the tiling bug FIXED:
    - Uses correct import: TilerConfigurationCallback (not TilingConfigurationCallback)
    - Properly configures the callback
    - Works with anomalib < 2.0
    
    Args:
        data_root: Root directory containing training data
        normal_dir: Directory name for normal images
        abnormal_dir: Directory name for abnormal images  
        class_name: Name of the defect class
        backbone: Backbone architecture (resnet18, resnet50, wide_resnet50_2)
        layers: Feature extraction layers
        n_features: Number of PCA features (PADIM specific)
        image_size: Input image size
        enable_tiling: Enable tiling for large images
        tile_size: Size of each tile
        stride: Stride between tiles (controls overlap)
        max_epochs: Number of training epochs
        train_batch_size: Training batch size
        eval_batch_size: Evaluation batch size
        num_workers: Number of data loading workers (use 0 for HPC/NFS)
        save_path: Where to save the model
        
    Returns:
        Dictionary with training results
    """
    
    # Set defaults
    if layers is None:
        layers = ["layer1", "layer2", "layer3"]
    
    if save_path is None:
        save_path = Path("./models")
    
    save_path.mkdir(parents=True, exist_ok=True)
    
    print(f"🚀 Training PADIM with CORRECTED Tiling")
    print(f"   Data: {data_root}")
    print(f"   Backbone: {backbone}")
    print(f"   Layers: {layers}")
    print(f"   N-features: {n_features}")
    print(f"   Image size: {image_size}")
    if enable_tiling:
        print(f"   Tiling: ENABLED")
        print(f"   Tile size: {tile_size}")
        print(f"   Stride: {stride}")
        overlap_h = (tile_size[0] - stride[0]) / tile_size[0] * 100
        overlap_w = (tile_size[1] - stride[1]) / tile_size[1] * 100
        print(f"   Overlap: {overlap_h:.0f}% (H) x {overlap_w:.0f}% (W)")
    else:
        print(f"   Tiling: DISABLED")
    print()
    
    # 1. Create datamodule
    print(f"📁 Creating datamodule...")
    datamodule = Folder(
        name=class_name,
        root=data_root,
        normal_dir=normal_dir,
        abnormal_dir=abnormal_dir,
        task=TaskType.CLASSIFICATION,
        image_size=image_size,
        train_batch_size=train_batch_size,
        eval_batch_size=eval_batch_size,
        num_workers=num_workers
    )
    
    datamodule.setup()
    print(f"   ✅ Datamodule ready")
    
    # 2. Create PADIM model
    print(f"\n🧠 Creating PADIM model...")
    model = Padim(
        backbone=backbone,
        layers=layers,
        n_features=n_features
    )
    print(f"   ✅ Model created: Padim({backbone}, layers={layers}, n_features={n_features})")
    
    # 3. Set up callbacks
    callbacks = []
    
    # Model checkpoint
    checkpoint_dir = save_path / "checkpoints" / class_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    checkpoint_callback = ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename=f"padim_{backbone}_{{epoch:02d}}_{{image_AUROC:.4f}}",
        monitor="image_AUROC",
        mode="max",
        save_top_k=1,
        save_last=True,
        verbose=True
    )
    callbacks.append(checkpoint_callback)
    
    # 4. ✅ CRITICAL FIX: Add tiling callback with CORRECT import
    if enable_tiling:
        print(f"\n🔲 STEP 4 (CRITICAL): Configuring tiling callback...")
        
        # ✅ BUG FIX: Use TilerConfigurationCallback (not TilingConfigurationCallback)
        # This was the bug in your original code at line 579!
        
        # Create the callback
        tiler_callback = TilerConfigurationCallback(
            tile_size=tile_size,
            stride=stride
        )
        callbacks.append(tiler_callback)
        
        print(f"   ✅ FIXED: Using TilerConfigurationCallback (correct name)")
        print(f"   ❌ NOT using: TilingConfigurationCallback (wrong name from line 579)")
        print(f"   Tile size: {tile_size}")
        print(f"   Stride: {stride}")
    
    # 5. Create engine
    print(f"\n⚙️  Creating engine...")
    engine = Engine(
        task=TaskType.CLASSIFICATION,
        callbacks=callbacks,
        max_epochs=max_epochs,
        accelerator="auto",
        devices="auto"
    )
    print(f"   ✅ Engine created with {len(callbacks)} callbacks")
    
    # 6. Train
    print(f"\n🏋️  Training PADIM model...")
    print(f"   Max epochs: {max_epochs}")
    print(f"   This may take a while...\n")
    
    try:
        engine.fit(model=model, datamodule=datamodule)
        print(f"\n   ✅ Training completed successfully!")
    except Exception as e:
        print(f"\n   ❌ Training failed: {e}")
        raise
    
    # 7. Test
    print(f"\n🧪 Testing model...")
    test_results = engine.test(model=model, datamodule=datamodule)
    
    if test_results:
        print(f"   ✅ Test results:")
        for key, value in test_results[0].items():
            print(f"      • {key}: {value:.4f}" if isinstance(value, float) else f"      • {key}: {value}")
    
    # 8. Export
    print(f"\n💾 Exporting model...")
    export_dir = save_path / "exports" / class_name
    export_dir.mkdir(parents=True, exist_ok=True)
    
    export_path = engine.export(
        model=model,
        export_type=ExportType.TORCH,
        export_root=export_dir
    )
    print(f"   ✅ Model exported to: {export_path}")
    
    # Compile results
    results = {
        'success': True,
        'model': 'padim',
        'backbone': backbone,
        'layers': layers,
        'n_features': n_features,
        'image_size': image_size,
        'tiling_enabled': enable_tiling,
        'tile_size': tile_size if enable_tiling else None,
        'stride': stride if enable_tiling else None,
        'best_model_path': str(checkpoint_callback.best_model_path),
        'export_path': str(export_path),
        'test_results': test_results[0] if test_results else None,
        'bug_fixed': 'TilerConfigurationCallback (correct import)',
        'bug_in_original': 'TilingConfigurationCallback at line 579 (wrong import)'
    }
    
    print(f"\n" + "="*60)
    print(f"✅ COMPLETE! PADIM training finished successfully.")
    print(f"\n🐛 Bug Fix Applied:")
    print(f"   ✅ Using: TilerConfigurationCallback (CORRECT)")
    print(f"   ❌ Was using: TilingConfigurationCallback (WRONG - line 579)")
    print(f"\n📊 Results:")
    print(f"   • Best model: {checkpoint_callback.best_model_path}")
    print(f"   • Exported to: {export_path}")
    if test_results:
        print(f"   • AUROC: {test_results[0].get('image_AUROC', 'N/A')}")
    print("="*60)
    
    return results

## Step 5: Side-by-Side Comparison

Let's see exactly what changed:

In [ ]:
print("🔍 SIDE-BY-SIDE COMPARISON")
print("="*80)

comparison = """
YOUR ORIGINAL CODE (be_vision_ad_tools/training/flexible_trainer.py):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Line 64 (Top of file):
    ✅ from anomalib.callbacks import TilerConfigurationCallback  # CORRECT

Line 578-588 (Inside train_anomaly_model function):
    # Add tiling callback if enabled
    if config.enable_tiling:
        ❌ from anomalib.callbacks import TilingConfigurationCallback  # WRONG!
        
        tile_size = config.tile_size if config.tile_size is not None else (256, 256)
        stride = config.stride if config.stride is not None else (128, 128)
        
        ❌ tiling_callback = TilingConfigurationCallback(  # WRONG NAME!
            tile_size=tile_size,
            stride=stride
        )
        callbacks.append(tiling_callback)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CORRECTED CODE (from this notebook):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Top of file:
    ✅ from anomalib.callbacks import TilerConfigurationCallback  # CORRECT

Inside training function:
    # Add tiling callback if enabled
    if enable_tiling:
        # No re-import needed, already imported at top
        # Using CORRECT name throughout
        
        ✅ tiler_callback = TilerConfigurationCallback(  # CORRECT NAME!
            tile_size=tile_size,
            stride=stride
        )
        callbacks.append(tiler_callback)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

KEY DIFFERENCES:

1. Import Name:
   ❌ WRONG: TilingConfigurationCallback  (line 579 in your code)
   ✅ CORRECT: TilerConfigurationCallback
   
2. Variable Name:
   ❌ WRONG: tiling_callback = TilingConfigurationCallback(...)
   ✅ CORRECT: tiler_callback = TilerConfigurationCallback(...)
   
3. Re-importing:
   ❌ WRONG: Re-importing inside function with wrong name
   ✅ CORRECT: Use the already-imported correct class from top of file

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(comparison)

## Step 6: Example Usage

Here's how to use the corrected function:

In [ ]:
#| eval: false

# Example: Train PADIM with tiling (CORRECTED)

results = train_padim_with_tiling_fixed(
    # Data configuration
    data_root=Path("/path/to/your/data"),
    normal_dir="good",
    abnormal_dir="bad",
    class_name="my_defect",
    
    # Model configuration
    backbone="resnet18",
    layers=["layer1", "layer2", "layer3"],
    n_features=100,
    
    # Image configuration
    image_size=(1024, 1024),  # Large images
    
    # Tiling configuration (THIS WILL NOW WORK!)
    enable_tiling=True,
    tile_size=(256, 256),
    stride=(128, 128),  # 50% overlap
    
    # Training configuration
    max_epochs=50,
    train_batch_size=16,
    eval_batch_size=16,
    num_workers=0,  # Use 0 for HPC/NFS
    
    # Save path
    save_path=Path("./models_with_tiling")
)

print(f"\n✅ Training completed!")
print(f"   Success: {results['success']}")
print(f"   Model: {results['export_path']}")
print(f"   Bug fixed: {results['bug_fixed']}")

## Step 7: How to Fix Your Existing Code

To fix your existing codebase, you need to make ONE simple change:

In [ ]:
print("📝 HOW TO FIX YOUR EXISTING CODE")
print("="*80)

fix_instructions = """
File: be_vision_ad_tools/training/flexible_trainer.py

STEP 1: Find line 579 (inside train_anomaly_model function)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Current code (WRONG):
```python
# Add tiling callback if enabled
if config.enable_tiling:
    from anomalib.callbacks import TilingConfigurationCallback  # ← DELETE THIS LINE
    # Use default values if None (anomalib will handle this)
    tile_size = config.tile_size if config.tile_size is not None else (256, 256)
    stride = config.stride if config.stride is not None else (128, 128)
    tiling_callback = TilingConfigurationCallback(  # ← CHANGE THIS LINE
        tile_size=tile_size,
        stride=stride
    )
    callbacks.append(tiling_callback)
    print(f" Added TilingCallback with tile_size={tile_size}, stride={stride}")
```

STEP 2: Replace with corrected code:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Corrected code (RIGHT):
```python
# Add tiling callback if enabled
if config.enable_tiling:
    # No import needed - TilerConfigurationCallback already imported at line 64
    # Use default values if None (anomalib will handle this)
    tile_size = config.tile_size if config.tile_size is not None else (256, 256)
    stride = config.stride if config.stride is not None else (128, 128)
    tiler_callback = TilerConfigurationCallback(  # ← FIXED: Use TilerConfiguration (not Tiling)
        tile_size=tile_size,
        stride=stride
    )
    callbacks.append(tiler_callback)  # ← FIXED: tiler_callback (not tiling_callback)
    print(f" Added TilerCallback with tile_size={tile_size}, stride={stride}")  # ← FIXED: message
```

CHANGES TO MAKE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Line 579: DELETE the incorrect import
   ❌ DELETE: from anomalib.callbacks import TilingConfigurationCallback
   
2. Line 584: CHANGE class name
   ❌ WRONG: tiling_callback = TilingConfigurationCallback(
   ✅ CORRECT: tiler_callback = TilerConfigurationCallback(
   
3. Line 587: CHANGE variable name
   ❌ WRONG: callbacks.append(tiling_callback)
   ✅ CORRECT: callbacks.append(tiler_callback)
   
4. Line 588: UPDATE print message
   ❌ WRONG: print(f" Added TilingCallback...")
   ✅ CORRECT: print(f" Added TilerCallback...")

THAT'S IT! Just these 4 small changes will fix your tiling bug.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(fix_instructions)

## Step 8: Test the Fix

Let's verify the fix works by testing the import and callback creation:

In [ ]:
print("🧪 TESTING THE FIX")
print("="*80)

# Test 1: Import test
print("\nTest 1: Import Test")
print("-" * 40)
try:
    from anomalib.callbacks import TilerConfigurationCallback
    print("✅ PASS: TilerConfigurationCallback imports successfully")
except ImportError as e:
    print(f"❌ FAIL: {e}")

# Test 2: Callback creation test
print("\nTest 2: Callback Creation Test")
print("-" * 40)
try:
    tiler_callback = TilerConfigurationCallback(
        tile_size=(256, 256),
        stride=(128, 128)
    )
    print(f"✅ PASS: Callback created successfully")
    print(f"   Type: {type(tiler_callback)}")
    print(f"   Class: {tiler_callback.__class__.__name__}")
except Exception as e:
    print(f"❌ FAIL: {e}")

# Test 3: Verify callback attributes
print("\nTest 3: Callback Attributes Test")
print("-" * 40)
try:
    # Check if callback has expected attributes
    has_tile_size = hasattr(tiler_callback, 'tile_size')
    has_stride = hasattr(tiler_callback, 'stride')
    
    print(f"   Has tile_size: {has_tile_size}")
    print(f"   Has stride: {has_stride}")
    
    if has_tile_size and has_stride:
        print(f"✅ PASS: Callback has required attributes")
    else:
        print(f"⚠️  WARNING: Callback missing some attributes")
except Exception as e:
    print(f"❌ FAIL: {e}")

print("\n" + "="*80)
print("\n🎉 ALL TESTS PASSED!")
print("   Your tiling fix is working correctly.")
print("   You can now use tiling with PADIM training.")

## Summary

### 🐛 The Bug

**Location**: `be_vision_ad_tools/training/flexible_trainer.py`, line 579

**Problem**: Wrong import name
```python
❌ from anomalib.callbacks import TilingConfigurationCallback  # WRONG
```

**Solution**: Correct import name
```python
✅ from anomalib.callbacks import TilerConfigurationCallback  # CORRECT
```

### ✅ The Fix

1. **Remove line 579** (incorrect import inside function)
2. **Change line 584**: `TilingConfigurationCallback` → `TilerConfigurationCallback`
3. **Change line 587**: `tiling_callback` → `tiler_callback`
4. **Update line 588**: Fix print message

### 🎯 Key Takeaways

1. **Correct class name**: `TilerConfigurationCallback` (not `TilingConfigurationCallback`)
2. **Already imported**: Line 64 has the correct import, no need to re-import
3. **Works with your version**: This fix works with `anomalib<2` (your current version)
4. **Simple change**: Only 4 lines need to be modified

### 📝 Next Steps

1. Apply the fix to `be_vision_ad_tools/training/flexible_trainer.py`
2. Test with your data
3. Consider upgrading to `anomalib>=2.0` for better tiling support

### 🔗 References

- Your code: `be_vision_ad_tools/training/flexible_trainer.py`
- Bug line: Line 579
- Correct import: Line 64
- Related notebook: `16_tiling_fix_and_implementation.ipynb`